# BERT Fine-Tuning: Twitter Sentiment Analysis
This notebook evaluates two strategies for sentiment classification:
1. **Frozen BERT:** Only the classification head is trained.
2. **Fine-Tuning (Last 2 Layers):** The last two encoder layers are unfrozen to adapt to specific Twitter linguistics.

In [1]:
!pip install transformers datasets evaluate accelerate -q

In [ ]:
import torch
import pandas as pd
import numpy as np
import evaluate
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

# 1. Hardware Setup (MUST SHOW 'cuda')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 2. Load Data (Twitter Dataset)
cols = ['id', 'entity', 'sentiment', 'text']
# We use engine='python' to avoid parsing errors with emojis/special chars
train_df = pd.read_csv('twitter_training.csv', header=None, names=cols, engine='python').dropna(subset=['text'])
valid_df = pd.read_csv('twitter_validation.csv', header=None, names=cols, engine='python').dropna(subset=['text'])

# Label Encoding
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['sentiment'])
valid_df['label'] = le.transform(valid_df['sentiment'])
num_labels = len(le.classes_)

# Split Validation into Val and Test (Requirement: Train, Val, Test sets)
val_df, test_df = train_test_split(valid_df, test_size=0.5, stratify=valid_df['label'], random_state=42)

# 3. Tokenization (bert-base-uncased)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_pandas(train_df[['text', 'label']]).map(tokenize_fn, batched=True)
val_ds = Dataset.from_pandas(valid_df[['text', 'label']]).map(tokenize_fn, batched=True)
test_ds = Dataset.from_pandas(test_df[['text', 'label']]).map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 4. Model Initialization Function (For Experiments)
def get_model(mode="last_two"):
    model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)

    if mode == "frozen":
        # EXPERIMENT: Freeze all BERT layers
        for param in model.bert.parameters():
            param.requires_grad = False
    elif mode == "last_two":
        # EXPERIMENT: Fine-tune only last 2 layers
        for param in model.bert.parameters():
            param.requires_grad = False
        for param in model.bert.encoder.layer[-2:].parameters():
            param.requires_grad = True

    return model.to(device)

# 5. Metrics Calculation (Handling Multi-class)
acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = acc_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

# 6. Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,          # Mandatory Learning Rate
    per_device_train_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",       
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True if torch.cuda.is_available() else False, # GPU speed boost
    report_to="none"
)

Using device: cuda


## Experiment 1: Frozen BERT Results

In [ ]:
# 7. Training Execution (Frozen experiment)
trainer_frozen = Trainer(
    model=get_model("frozen"),
    args=training_args,
    train_dataset=train_ds.shuffle(seed=42).select(range(5000)), # sampled for speed
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_frozen.train()

# 8. Evaluation
print("\n--- FROZEN MODEL PERFORMANCE ---")
output = trainer_frozen.predict(test_ds)
y_preds = np.argmax(output.predictions, axis=-1)
y_true = output.label_ids
print(classification_report(y_true, y_preds, target_names=le.classes_))

cm = confusion_matrix(y_true, y_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: Twitter Sentiment (Frozen 2 Layers)')
plt.show()

--- FROZEN MODEL PERFORMANCE ---
              precision    recall  f1-score   support

  Irrelevant       0.00      0.00      0.00        86
    Negative       0.34      0.32      0.33       133
     Neutral       0.50      0.01      0.03       143
    Positive       0.30      0.81      0.44       138

    accuracy                           0.31       500
   macro avg       0.28      0.29      0.20       500
weighted avg       0.32      0.31      0.22       500


## Experiment 2: Fine-Tuning Results & Confusion Matrix

In [ ]:
# 7. Training Execution (Last 2 layers experiment)
trainer = Trainer(
    model=get_model("last_two"),
    args=training_args,
    train_dataset=train_ds.shuffle(seed=42).select(range(5000)),
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# 8. Evaluation & Confusion Matrix
print("\n--- FINAL TEST SET PERFORMANCE ---")
output = trainer.predict(test_ds)
y_preds = np.argmax(output.predictions, axis=-1)
y_true = output.label_ids

print(classification_report(y_true, y_preds, target_names=le.classes_))

cm = confusion_matrix(y_true, y_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: Twitter Sentiment (Last 2 Layers)')
plt.show()


--- FINE-TUNED MODEL PERFORMANCE ---
              precision    recall  f1-score   support

  Irrelevant       0.67      0.05      0.09        86
    Negative       0.53      0.74      0.62       133
     Neutral       0.56      0.59      0.57       143
    Positive       0.54      0.62      0.57       138

    accuracy                           0.54       500
   macro avg       0.57      0.50      0.46       500
weighted avg       0.56      0.54      0.50       500
